In [1]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
pd.set_option('display.max_columns', None)
import time
import warnings
warnings.filterwarnings("ignore")
import os
import sys

In [3]:
from illume_plus import data_utils

# Prepare one drug from the transcriptomics file

Select a drug ID, load its original regression labels and gene features, and save the three processed datasets for the main example.


In [4]:
# drug_folder_number = 1010 # Gefitinib
# drug_folder_number = 1011 # Navitoclax
# drug_folder_number = 1051 # Alisertib
# drug_folder_number = 1909 # Venetoclax
drug_folder_number = 1168 # Erlotinib

In [5]:
from pathlib import Path

data_dir = Path.cwd() / "data"
if not data_dir.is_dir():
    data_dir = Path.cwd().parent / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or examples directory")
base_data_path = str(data_dir.resolve())
full_path_columns = f"{base_data_path}/gdsc/{drug_folder_number}/splits/columns_Boruta.csv"
full_path_trasc = f"{base_data_path}/transcriptomics.feather"
full_path_splits = f"{base_data_path}/gdsc/{drug_folder_number}/splits/"


In [6]:
# Load transcriptomics data once
ordered_features = data_utils.load_feature_order(full_path_columns) # this is the list of genes found in the file columns
transcriptomics_all = data_utils.load_transcriptomics_data(full_path_trasc)  # this is a df that associates ALL genes to all cell lines
transcriptomics_indexed = data_utils.filter_transcriptomics_by_genes(transcriptomics_all, ordered_features) #filter the transcriptomics df
feature_names = np.array(ordered_features)

Loading feature order from ../data//gdsc/1168/splits/columns_Boruta.csv...
Number of ordered features: 1134
Loading transcriptomics data from ../data//transcriptomics.feather...
Transcriptomics shape: (14197, 18176)
Filtering transcriptomics data by top genes...
Transcriptomics columns: 18174
Top genes found: 1134 out of 1134
Filtered transcriptomics shape: (14197, 1134)
Original shape was: (14197, 18174)


In [7]:
# train dataset
train_data = data_utils.process_dataset('train', full_path_splits, transcriptomics_indexed, feature_names, 
                            fit_scaler=True)

# Process test set (use training scaler)
test_data = data_utils.process_dataset('test', full_path_splits, transcriptomics_indexed, feature_names, 
                        scaler=train_data['scaler'], fit_scaler=False) 

# Process validation set (use training scaler)  
val_data = data_utils.process_dataset('valid', full_path_splits, transcriptomics_indexed, feature_names, 
                            scaler=train_data['scaler'], fit_scaler=False)


=== PROCESSING TRAIN DATASET ===
Loading instance names from ../data//gdsc/1168/splits/train_X.csv...
Loaded 583 instances
First few instances: ['ACH-001711' 'ACH-000087' 'ACH-000644']
Loading labels/predictions from ../data//gdsc/1168/splits/train_Y.csv...
Loaded 583 labels/predictions
First few values: [-0.5232259  -0.91616097  0.73936869]
Creating feature matrix...
Columns entries found among genes: 1134 out of 1134
All cell lines (instances) in splits are in transcriptomics
Feature matrix shape: (583, 1134)
Applying standard scaling...
Before scaling - Mean: 0.0015, Std: 1.0686
Fitted new scaler on training data
After scaling - Mean: -0.0000, Std: 1.0000
Final train dataset - X: (583, 1134), y: (583,)

=== PROCESSING TEST DATASET ===
Loading instance names from ../data//gdsc/1168/splits/test_X.csv...
Loaded 48 instances
First few instances: ['ACH-001334' 'ACH-000075' 'ACH-000021']
Loading labels/predictions from ../data//gdsc/1168/splits/test_Y.csv...
Loaded 48 labels/predictions


In [ ]:
# Save the original regression labels, scaled features, feature names, and scaler.
import gzip
import pickle

output_path = data_dir / "gdsc" / str(drug_folder_number) / "processed.pkl.gz"
with gzip.open(output_path, "wb", compresslevel=6) as file:
    pickle.dump(
        {"train": train_data, "val": val_data, "test": test_data},
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
print(f"Saved {output_path}")
